In [157]:
import math
import locale
import numpy as np
import mip
from collections import namedtuple
import glob
import pandas as pd
from IPython.display import display, Markdown

In [158]:
##### Open the main sheet #####

directory = "examples/Mariners"
globFilename = directory + "/Full Name*.xls*"
excelFiles = glob.glob(globFilename)
if len(excelFiles) < 1 or len(excelFiles) > 1:
    print(f"Can't find unique excel file: {globFilename}")
    exit()
mainSheet = pd.read_excel(excelFiles[0])

In [159]:
##### Find the constraint row #####
for constraintRow in range(len(mainSheet.Date)):
    if mainSheet.iat[constraintRow,2] == "At least":
        break

In [160]:
##### Establish an object for each day of the season
 
locale.setlocale(locale.LC_ALL, '')
Game = namedtuple('Game', ('weekday', 'month', 'day', 'gameDay', 'time', 'opponent', 'type', 'price', 'pairs', 'seats'))

##### Read the season schedule and store it as a list of games
 
openingDay = mainSheet.Date[0]
schedule = []
GamesInPlan = 0
PairsInPlan = 0
MaxPairsPerGame = 0
for weekday, date, time, opponent, gameType, price, pairs, seats in zip(mainSheet.Day, mainSheet.Date, mainSheet.Time, mainSheet.Opponent, mainSheet.Type, mainSheet.Price, mainSheet.GamePairs, mainSheet.Seats):
    if not isinstance(weekday, str) or weekday == "" or pd.isna(date):
        break
    schedule.append(Game(weekday, date.month, date.day, (date - openingDay).days, time.strftime("%I:%M %p"), opponent, gameType, price, pairs, seats))
    GamesInPlan += 1
    PairsInPlan += pairs
    MaxPairsPerGame = max(MaxPairsPerGame, pairs)


In [161]:
#########################################################
# Build a dictionary out of a list of games
#
#     pairsOrQuads == 0:    Create constraints for pair and quads
#     pairsOrQuads == 2:    Create constraint for pairs only
#     pairsOrQuads == 4:    Create constraint for quads only

def BuildDict(gameList, pairsOrQuads = 0):
    pairList = []
    for game in gameList:
        if pairsOrQuads != 4:
            pairList.append((game, 1.0))
        if pairsOrQuads != 2:
            pairList.append((game + GamesInPlan, 1.0))
    return dict(pairList)

# Define a class to contain constraints

class Constraint:
    def __init__(self, description, isChecked, comparator, value, gameDictionaries = []):
        self.description = description
        self.isChecked = isChecked
        self.comparator = comparator
        self.value = value
        self.gameDictionaries = gameDictionaries

# Define a class to contain each person's preferences

class SportsFan:
    def __init__(self, name, pairs, quads, ranking, extra = []):
        self.name = name
        self.pairs = pairs
        self.quads = quads

# Assign weights to games

        if len(ranking) != 0:
            self.ranking = ranking
        else:
            self.ranking = GamesInPlan * [GamesInPlan // 2]

# Adjust weights to favor highly ranked games

        self.useRanking = []
        midpoint = (GamesInPlan - 1) // 2
        for wgt in self.ranking:
            if wgt <= midpoint + 1:
                self.useRanking.append(math.sqrt(wgt - 1.0))
            else:
                self.useRanking.append(2.0 * math.sqrt(midpoint) - math.sqrt(2.0 * midpoint - wgt + 1))
        if len(self.useRanking) == GamesInPlan:
            self.useRanking += [2.0 * cost for cost in self.useRanking]
        else:
            for ix in range(GamesInPlan):
                self.useRanking[GamesInPlan + ix] *= 2

# Each person must attend correct number of games

        pairCon = Constraint(f"{self.name}: Pairs = {self.pairs}", True, '==', self.pairs, [dict([(ix, 1.0) for ix in range(GamesInPlan)])])
        quadCon = Constraint(f"{self.name}: Quads = {self.quads}", True, '==', self.quads, [dict([(ix, 1.0) for ix in range(GamesInPlan, 2 * GamesInPlan)])])

# Save all of the constraints for this person

        self.constraints = [pairCon, quadCon] + extra

##### This constraint handles the spacing of games

def Spacing(description, checked, comparator, value, pairsOrQuads = 0):
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return None
    daysApart = value + 1
    constraint = Constraint(description, checked, comparator, np.float64(1.0))
    firstGame = 0
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].gameDay + daysApart > schedule[lastGame].gameDay:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        while schedule[firstGame].gameDay + daysApart <= schedule[lastGame].gameDay:
            firstGame += 1
    return constraint

##### Require or forbid games in various months

def Monthly(description, checked, comparator, value, pairsOrQuads = 0):
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > 30:
        print(f"Bad constraint:{description}")
        return None
    constraint = Constraint(description, checked, comparator, value)
    firstGame = 0
    lastGame = 0
    while True:
        while schedule[lastGame].month < 5:
            lastGame += 1
        while lastGame < GamesInPlan and schedule[firstGame].month == schedule[lastGame].month:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraint

##### Require or forbid games in different series

def Series(description, checked, comparator, value):
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return None
    constraint = Constraint(description, checked, comparator, value)
    firstGame = 0
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame)))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraint

##### Require or forbid games for different opponents

def Opponents(description, checked, comparator, value):
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return None
    constraint = Constraint(description, checked, comparator, value)
    firstGame = 0
    lastGame = 0
    opponentDict = {}
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        opponentGames = opponentDict.get(schedule[firstGame].opponent, [])
        opponentGames += range(firstGame, lastGame)
        opponentDict[schedule[firstGame].opponent] = opponentGames
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    for opponentGames in opponentDict.values():
        constraint.gameDictionaries.append(BuildDict(opponentGames))
    return constraint

In [162]:
##### Define the participants here #####

fans = []
totalPairs = 0
for fullName, nPairs, nQuads in zip(mainSheet.FullName, mainSheet.Pairs, mainSheet.Quads):
    if not isinstance(fullName, str) or fullName == "":
        break
    totalPairs += nPairs + 2 * nQuads
    globFilename = directory + "/" + fullName + "*.xls*"
    excelFiles = glob.glob(globFilename)
    if len(excelFiles) < 1 or len(excelFiles) > 1:
        print(f"Can't find unique excel file: {globFilename}")
        continue
    fanSheet = pd.read_excel(excelFiles[0])
    pairsRank = [np.int64(fanSheet.Pick[ix]) for ix in range(GamesInPlan)]
    if not isinstance(fanSheet.QuadPick[0], np.float64) and not math.isnan(fanSheet.QuadPick[0]):
        quadsRank = [np.int64(fanSheet.QuadPick[ix]) for ix in range(GamesInPlan)]
        pairsRank += quadsRank

    # Add fan constraints
    extraConstraints = []
    row = constraintRow
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Monthly(description, fanSheet.iat[row,1], '>=', fanSheet.iat[row, 3]))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Monthly(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3]))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Monthly(description, fanSheet.iat[row,1], '>=', fanSheet.iat[row, 3], 2))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Monthly(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3], 2))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Monthly(description, fanSheet.iat[row,1], '>=', fanSheet.iat[row, 3], 4))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Monthly(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3], 4))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Spacing(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3]))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Spacing(description, fanSheet.iat[row,1], '>=', fanSheet.iat[row, 3]))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Spacing(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3], 2))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Spacing(description, fanSheet.iat[row,1], '>=', fanSheet.iat[row, 3], 2))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Spacing(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3], 4))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Spacing(description, fanSheet.iat[row,1], '>=', fanSheet.iat[row, 3], 4))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Series(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3]))
    row += 1
    if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
        description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
        extraConstraints.append(Opponents(description, fanSheet.iat[row,1], '<=', fanSheet.iat[row, 3]))
    fans.append(SportsFan(fullName, nPairs, nQuads, pairsRank, extraConstraints))

leftOver = PairsInPlan - totalPairs
if leftOver < 0:
    print("Too many games requested")
maxSparePairs = max((leftOver * GamesInPlan) // PairsInPlan, 1) # Maximum # of games that could be completely unassigned
while leftOver > 0:
    pairsRank = (GamesInPlan * [np.int64(1)])[:]
    nPairs = min(leftOver, maxSparePairs)
    fans.append(SportsFan(f"Spare Pair", nPairs, 0, pairsRank))
    leftOver -= nPairs


In [163]:
for gix, game in enumerate(schedule):
    picks = []
    for fan in fans:
        picks.append(int(fan.ranking[gix]))
    print(f"{picks} {game.weekday} {game.month}/{game.day} {game.opponent}{game.time} {game.type} {game.seats}")

[19, 63, 1, 1, 1] Thu 3/26 Guardians 07:10 PM A $90.00 (4)
[55, 30, 2, 1, 1] Fri 3/27 Guardians 06:45 PM B $69.00 (4)
[62, 36, 3, 1, 1] Sat 3/28 Guardians 06:40 PM B $69.00 (4)
[34, 28, 4, 1, 1] Sun 3/29 Guardians 04:20 PM B $69.00 (4)
[66, 29, 5, 1, 1] Mon 3/30 Yankees 06:40 PM D $45.00 (4)
[15, 9, 6, 1, 1] Tue 3/31 Yankees 06:40 PM D $45.00 (4)
[63, 31, 7, 1, 1] Wed 4/1 Yankees 01:10 PM D $45.00 (4)
[39, 42, 8, 1, 1] Fri 4/10 Astros 06:40 PM D $45.00 (4)
[69, 19, 9, 1, 1] Sat 4/11 Astros 06:40 PM D $45.00 (4)
[38, 43, 10, 1, 1] Sun 4/12 Astros 01:10 PM D $45.00 (4)
[68, 21, 11, 1, 1] Mon 4/13 Astros 01:10 PM B $69.00 (4)
[22, 3, 12, 1, 1] Fri 4/17 Rangers 06:40 PM B $69.00 (4)
[31, 77, 13, 1, 1] Sat 4/18 Rangers 04:15 PM B $69.00 (4)
[40, 35, 14, 1, 1] Sun 4/19 Rangers 01:10 PM B $69.00 (4)
[54, 65, 15, 1, 1] Mon 4/20 Athletics 06:40 PM B $69.00 (4)
[6, 46, 16, 1, 1] Tue 4/21 Athletics 06:40 PM B $69.00 (4)
[51, 68, 17, 1, 1] Wed 4/22 Athletics 01:10 PM D $45.00 (4)
[58, 73, 18, 1, 1

In [164]:
try:
    tixModel = mip.Model()
    tixVars = [tixModel.add_var(var_type = mip.BINARY) for ix in range(2 * GamesInPlan * len(fans))]

    # All tickets must be allocated

    for ix in range(GamesInPlan):
        tixModel += mip.xsum(tixVars[ix + 2 * iy * GamesInPlan] + 2.0 * tixVars[ix + GamesInPlan + 2 * iy * GamesInPlan] for iy in range(len(fans))) == schedule[ix].pairs

    # Each fan must attend the correct number of games + satisfy all personal constraints

    for iy, fan in enumerate(fans):
        for constraint in fan.constraints:
            if constraint is not None:
                for coefDict in constraint.gameDictionaries:
                    linfunc = mip.xsum(count * tixVars[ix + iy * 2 * GamesInPlan] for ix, count in coefDict.items())
                    if constraint.comparator == '==':
                        tixModel += linfunc == constraint.value
                    if constraint.comparator == '<=':
                        tixModel += linfunc <= constraint.value
                    if constraint.comparator == '>=':
                        tixModel += linfunc >= constraint.value

    # Establish the objective function

    costs = []
    for fan in fans:
        costs += fan.useRanking
    tixModel.objective = mip.xsum(costs[ix] * tixVars[ix] for ix in range(len(tixVars)))
except Exception as e:
    print(f"Mip setup exception: {e}")

In [165]:
try:
    status = tixModel.optimize()
except Exception as e:
    print(f"Mip optimize exception: {e}")

if status != mip.OptimizationStatus.OPTIMAL:
    print(f"No solution found: {status}")

In [166]:
picks = [[] for fan in fans]
for mix, mvar in enumerate(tixModel.vars):
    if mvar.x is not None and mvar.x > 0.5:
        fix = mix // (2 * GamesInPlan)
        if len(fans[fix].ranking) > GamesInPlan:
            gix = mix % (2 * GamesInPlan)
        else:
            gix = mix % GamesInPlan
        picks[fix].append(fans[fix].ranking[gix])
for fix, fan in enumerate(fans):
    picks[fix].sort()
    picks[fix] = [int(pick) for pick in picks[fix]]
    print(fans[fix].name, picks[fix])

Tom Grandine [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Al Erisman [1, 1, 2, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Eric Brechner [3, 5, 8, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 64, 66]
Spare Pair [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Spare Pair [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [167]:
costs = {}
gameAllocationTable = "Day | Date | Time | Opponent | Type | Seats |"
for ix in range(MaxPairsPerGame):
    gameAllocationTable += f" Pair{ix+1} |"
gameAllocationTable += "\n| :-: | :-: | :-: | :-: | :-: | :-: |"  + " :-: |" * MaxPairsPerGame + "\n"
for gix, game in enumerate(schedule):
    gameAllocationTable += f"| {game.weekday} | {game.month}/{game.day} | {game.time} | {game.opponent} | {game.type} | {game.seats} |"
    for mix, mvar in enumerate(tixModel.vars):
        if mix % GamesInPlan == gix and mvar.x is not None and mvar.x > 0.5:
            fix = mix // (2 * GamesInPlan)
            if len(fans[fix].ranking) > GamesInPlan and mix % (2 * GamesInPlan) >= GamesInPlan:
                gix += GamesInPlan
            cost = costs.get(fans[fix].name, 0.0)
            costs[fans[fix].name] = cost + 2.0 * game.price
            gameAllocationTable += f" {fans[fix].name} ({fans[fix].ranking[gix]}) |"
            if mix % (2 * GamesInPlan) >= GamesInPlan:
                costs[fans[fix].name] += 2.0 * game.price
                gameAllocationTable += " |"
    gameAllocationTable += " ❌ |" * (MaxPairsPerGame - game.pairs) + "\n"
display(Markdown(gameAllocationTable))

Day | Date | Time | Opponent | Type | Seats | Pair1 | Pair2 |
| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| Thu | 3/26 | 07:10 PM | Guardians  | A | $90.00 (4) | Al Erisman (1) | |
| Fri | 3/27 | 06:45 PM | Guardians  | B | $69.00 (4) | Al Erisman (2) | |
| Sat | 3/28 | 06:40 PM | Guardians  | B | $69.00 (4) | Eric Brechner (3) | Spare Pair (1) |
| Sun | 3/29 | 04:20 PM | Guardians  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 3/30 | 06:40 PM | Yankees  | D | $45.00 (4) | Eric Brechner (5) | Spare Pair (1) |
| Tue | 3/31 | 06:40 PM | Yankees  | D | $45.00 (4) | Al Erisman (9) | Spare Pair (1) |
| Wed | 4/1 | 01:10 PM | Yankees  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 4/10 | 06:40 PM | Astros  | D | $45.00 (4) | Eric Brechner (8) | Spare Pair (1) |
| Sat | 4/11 | 06:40 PM | Astros  | D | $45.00 (4) | Al Erisman (19) | Spare Pair (1) |
| Sun | 4/12 | 01:10 PM | Astros  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 4/13 | 01:10 PM | Astros  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 4/17 | 06:40 PM | Rangers  | B | $69.00 (4) | Al Erisman (3) | Eric Brechner (12) |
| Sat | 4/18 | 04:15 PM | Rangers  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 4/19 | 01:10 PM | Rangers  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 4/20 | 06:40 PM | Athletics  | B | $69.00 (4) | Eric Brechner (15) | Spare Pair (1) |
| Tue | 4/21 | 06:40 PM | Athletics  | B | $69.00 (4) | Tom Grandine (6) | Spare Pair (1) |
| Wed | 4/22 | 01:10 PM | Athletics  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 5/1 | 06:45 PM | Royals  | D | $45.00 (4) | Eric Brechner (18) | Spare Pair (1) |
| Sat | 5/2 | 06:40 PM | Royals  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 5/3 | 01:10 PM | Royals  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 5/4 | 06:40 PM | Braves  | A | $90.00 (4) | Eric Brechner (21) | Spare Pair (1) |
| Tue | 5/5 | 06:40 PM | Braves  | B | $69.00 (4) | Al Erisman (6) | Spare Pair (1) |
| Wed | 5/6 | 01:10 PM | Braves  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 5/15 | 06:40 PM | Padres  | B | $69.00 (4) | Tom Grandine (11) | Eric Brechner (24) |
| Sat | 5/16 | 04:15 PM | Padres  | D | $45.00 (4) | Tom Grandine (2) | |
| Sun | 5/17 | 04:20 PM | Padres  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 5/18 | 06:40 PM | White Sox  | D | $45.00 (4) | Al Erisman (7) | Eric Brechner (27) |
| Tue | 5/19 | 06:40 PM | White Sox  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 5/20 | 01:10 PM | White Sox  | B | $69.00 (4) | Al Erisman (8) | Spare Pair (1) |
| Fri | 5/29 | 07:10 PM | D-backs  | B | $69.00 (4) | Al Erisman (11) | Eric Brechner (30) |
| Sat | 5/30 | 07:10 PM | D-backs  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 5/31 | 01:10 PM | D-backs  | C | $57.00 (4) | Al Erisman (18) | Spare Pair (1) |
| Mon | 6/1 | 06:40 PM | Mets  | C | $57.00 (4) | Eric Brechner (33) | Spare Pair (1) |
| Tue | 6/2 | 06:40 PM | Mets  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 6/3 | 12:40 PM | Mets  | A | $90.00 (4) | Tom Grandine (12) | Spare Pair (1) |
| Tue | 6/16 | 06:40 PM | Orioles  | A | $90.00 (4) | Eric Brechner (36) | Spare Pair (1) |
| Wed | 6/17 | 06:40 PM | Orioles  | B | $69.00 (4) | Al Erisman (5) | Spare Pair (1) |
| Thu | 6/18 | 01:10 PM | Orioles  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 6/19 | 07:10 PM | Red Sox  | B | $69.00 (4) | Eric Brechner (39) | Spare Pair (1) |
| Sat | 6/20 | 07:10 PM | Red Sox  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 6/21 | 01:10 PM | Red Sox  | C | $57.00 (4) | Tom Grandine (4) | Spare Pair (1) |
| Mon | 6/29 | 06:40 PM | Angels  | C | $57.00 (4) | Al Erisman (15) | Eric Brechner (42) |
| Tue | 6/30 | 06:40 PM | Angels  | B | $69.00 (4) | Al Erisman (1) | Spare Pair (1) |
| Thu | 7/2 | 06:40 PM | Angels  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 7/3 | 07:10 PM | Blue Jays  | A | $90.00 (4) | Eric Brechner (45) | Spare Pair (1) |
| Sat | 7/4 | 01:10 PM | Blue Jays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 7/5 | 02:00 PM | Blue Jays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 7/17 | 07:10 PM | Giants  | A | $90.00 (4) | Al Erisman (13) | Eric Brechner (48) |
| Sat | 7/18 | 05:08 PM | Giants  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 7/19 | 01:10 PM | Giants  | C | $57.00 (4) | Tom Grandine (9) | Spare Pair (1) |
| Mon | 7/20 | 06:40 PM | Reds  | C | $57.00 (4) | Eric Brechner (51) | Spare Pair (1) |
| Tue | 7/21 | 06:40 PM | Reds  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 7/22 | 12:40 PM | Reds  | B | $69.00 (4) | Tom Grandine (7) | Spare Pair (1) |
| Fri | 7/31 | 07:10 PM | Twins  | A | $90.00 (4) | Eric Brechner (54) | Spare Pair (1) |
| Sat | 8/1 | 01:10 PM | Twins  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 8/2 | 01:10 PM | Twins  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Tue | 8/4 | 06:40 PM | Tigers  | C | $57.00 (4) | Eric Brechner (57) | Spare Pair (1) |
| Wed | 8/5 | 06:40 PM | Tigers  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Thu | 8/6 | 01:10 PM | Tigers  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 8/7 | 07:10 PM | Rays  | A | $90.00 (4) | Eric Brechner (60) | Spare Pair (1) |
| Sat | 8/8 | 06:50 PM | Rays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 8/9 | 01:10 PM | Rays  | A | $90.00 (4) | Tom Grandine (1) | |
| Fri | 8/21 | 07:10 PM | Cubs  | B | $69.00 (4) | Tom Grandine (5) | Al Erisman (10) |
| Sat | 8/22 | 06:40 PM | Cubs  | B | $69.00 (4) | Eric Brechner (64) | Spare Pair (1) |
| Sun | 8/23 | 01:10 PM | Cubs  | B | $69.00 (4) | Al Erisman (14) | Spare Pair (1) |
| Mon | 8/24 | 06:40 PM | Phillies  | B | $69.00 (4) | Tom Grandine (3) | Eric Brechner (66) |
| Tue | 8/25 | 06:40 PM | Phillies  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 8/26 | 01:10 PM | Phillies  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Thu | 9/3 | 06:40 PM | Athletics  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 9/4 | 07:10 PM | Athletics  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 9/5 | 06:40 PM | Athletics  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 9/6 | 01:10 PM | Athletics  | C | $57.00 (4) | Al Erisman (4) | Spare Pair (1) |
| Tue | 9/8 | 06:40 PM | Rangers  | B | $69.00 (4) | Al Erisman (16) | Spare Pair (1) |
| Wed | 9/9 | 06:40 PM | Rangers  | B | $69.00 (4) | Al Erisman (2) | Spare Pair (1) |
| Thu | 9/10 | 01:10 PM | Rangers  | B | $69.00 (4) | Al Erisman (17) | Spare Pair (1) |
| Tue | 9/22 | 06:40 PM | Astros  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 9/23 | 07:10 PM | Astros  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Thu | 9/24 | 06:40 PM | Angels  | C | $57.00 (4) | Tom Grandine (8) | Spare Pair (1) |
| Fri | 9/25 | 07:10 PM | Angels  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 9/26 | 06:40 PM | Angels  | A | $90.00 (4) | Tom Grandine (10) | Al Erisman (20) |
| Sun | 9/27 | 12:10 PM | Angels  | A | $90.00 (4) | Al Erisman (12) | Spare Pair (1) |


In [168]:
amountOwedTable = "| | Amount owed | Paid |\n"
amountOwedTable += "| :- | -: | -: |\n"
for name, cost in sorted(costs.items()):
    amountOwedTable += f"| {name} | {locale.currency(cost, grouping=True)} | |\n"
display(Markdown(amountOwedTable))

| | Amount owed | Paid |
| :- | -: | -: |
| Al Erisman | $3,306.00 | |
| Eric Brechner | $3,000.00 | |
| Spare Pair | $13,998.00 | |
| Tom Grandine | $1,932.00 | |
